# Import

In [ ]:
%load_ext autoreload
%autoreload 2

import random

from deap import base, creator, tools

import config.params as params
from config.params import Sensor, Gene, Individual, Population
import config.seeding as seeding
import config.sensors as sensors

# Initialization

In [ ]:
# Define the fitness to maximize
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Define the Individual class as a list of Gene objects with an associated fitness.
creator.create("Individual", list, fitness=creator.FitnessMax)  # type: ignore

toolbox = base.Toolbox()

In [ ]:
import custom_toolbox.initialize.initialize as initialize

# Register the individual generator.
toolbox.register("individual", initialize.create_individual, creator.Individual)    # type: ignore

# Register the population generator (creates a list of individuals).
toolbox.register("population", tools.initRepeat, list, toolbox.individual)  # type: ignore

# NOTE: Consider Demes
# A deme is a sub-population that is contained in a population.
# https://deap.readthedocs.io/en/master/tutorials/basic/part1.html#demes

# Create initial population.
population = initialize.create_seeded_population(
    creator.Individual, # type: ignore
    toolbox.individual, # type: ignore
    seeding.SEED_INDIVIDUALS,
    params.POPULATION_SIZE,
)

In [ ]:
import utils.visualization as visualization

visualization.visualize_population(population, max_display=10)

# Evaluation

In [ ]:
import custom_toolbox.evaluate.evaluate_fitness_mock as evaluate

# Register the custom evaluation function (use the standard 'evaluate' name).
toolbox.register("evaluate", evaluate.evaluate_individual)

# Selection

In [ ]:
# Tournament selection with tournament size of 3.
toolbox.register("select", tools.selTournament, tournsize=3)

# Mutation

In [ ]:
import custom_toolbox.mutate.mutate as mutate

# Register the custom mutation function.
toolbox.register("mutate", mutate.mutate_sensor_layout)

# Mating

In [ ]:
import custom_toolbox.mate.mate as mate

# Register the custom crossover function.
toolbox.register("mate", mate.cx_variable_length_bounded)

# Evolution

In [ ]:
import numpy as np
from deap import tools

# 1. Create a Statistics object that looks at the first objective values
stats = tools.Statistics(key=lambda ind: ind.fitness.values[0])

# 2. Register the metrics you want to track
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

# 3. Create a Logbook to store the data nicely
logbook = tools.Logbook()
logbook.header = ["gen", "nevals"] + (stats.fields if stats else [])

In [ ]:
hof = tools.HallOfFame(1) # Remembers the 1 absolute best individual

# Evaluate the entire population (initial baseline)
invalid_ind = [ind for ind in population if not ind.fitness.valid]
fitnesses = list(toolbox.map(toolbox.evaluate, invalid_ind))
for ind, fit in zip(invalid_ind, fitnesses):
    ind.fitness.values = fit

# Update the Hall of Fame with the initial baseline
hof.update(population)

for gen in range(params.NGEN):
    print(f"-- Generation {gen} --")
    print(f"Population size: {len(population)}")

    # Extract the elites BEFORE selection
    elites = tools.selBest(population, params.ELITE_COUNT)
    elites = list(map(toolbox.clone, elites))

    # Select the rest of the parents (Population Size - Elite Count)
    offspring = toolbox.select(population, len(population) - params.ELITE_COUNT)
    offspring = list(map(toolbox.clone, offspring))

    # Apply crossover on the offspring
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < 0.7:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    # Apply mutation on the offspring
    for mutant in offspring:
        toolbox.mutate(mutant)
        del mutant.fitness.values

    # Evaluate the individuals with an invalid fitness
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    # The population is entirely replaced by the offspring + elites
    population[:] = offspring + elites

    hof.update(population)

    # --- NEW STATS TRACKING ---
    # Compute the statistics for the current population
    record = stats.compile(population)
    
    # Record the data in the logbook (nevals is the number of newly evaluated individuals)
    logbook.record(gen=gen, nevals=len(invalid_ind), **record)
    
    # Print the logbook output for this generation
    print(logbook.stream)

print("Best overall layout found:", hof[0])
print("Number of sensors in best layout:", len(hof[0]))
print("Best overall fitness:", hof[0].fitness.values)

In [ ]:
import utils.visualization as visualization

visualization.visualize_evolution(logbook)